# ES4304 — Environment Setup & Check

**Run this notebook once before your first tutorial.** It confirms that your environment has everything the course needs, and draws a test map.

## Where you are running

The course runs in a **GitHub Codespace**: the environment is pre-built and this repository is already here, so there is nothing to install. The setup cell below only checks — it installs nothing.

If you are running locally instead, you built the environment once from `environment.yml`; see the [module README](README.md).

**Stop your codespace when you finish** — *Codespaces → Stop codespace*, or close it from <https://github.com/codespaces>. It keeps billing against your monthly allowance while it idles.

## 1. Setup and environment check

Run the cell below. It:

1. Reads the package list from [`environment.yml`](environment.yml).
2. Imports each package and prints its version.
3. Prints your Python version and working directory.

Every line should say `OK`. If any does not, the repair cell in 1.1 fixes it.

**Why it reads `environment.yml`.** That one file builds the Codespaces image and a local conda env. Checking against the same file means this cell cannot drift from what was installed: add a package there and it is checked here, with no edit to this notebook.

In [ ]:
# --- Setup: run this cell first. Safe to re-run. ---------------------------
# environment.yml is the single source of truth for the environment: it builds
# the Codespaces image and a local conda env, and this cell checks against the
# same file. If something is wrong, the repair cell below fixes it in place.
import importlib
import os
import platform
import re

# The working directory is the notebook's own folder, so environment.yml is a
# sibling. The second path catches a Jupyter started at the repository root.
ENV_CANDIDATES = ["environment.yml",
                  os.path.join("0.1_Setup_and_Environment", "environment.yml")]

# conda name -> import name, for the few that differ. matplotlib-base is
# matplotlib without the Qt GUI backends; netcdf4 installs a module spelled
# netCDF4; scikit-image is imported as skimage.
IMPORT_NAME = {"matplotlib-base": "matplotlib",
               "netcdf4": "netCDF4",
               "scikit-image": "skimage"}

# Listed in environment.yml, but not something a notebook imports: the
# interpreter itself, and the tooling that runs the notebook.
NOT_IMPORTED = {"python", "jupyterlab", "ipykernel"}


def required_packages(path):
    """Package names from the dependencies: block of an environment.yml.

    Regex rather than PyYAML: the block is a flat list, and pyyaml is not in
    environment.yml, so importing it would add one more thing that can be
    missing in the very environment this cell is checking.
    """
    names, in_deps = [], False
    with open(path) as fh:
        for line in fh:
            if line.startswith("dependencies:"):
                in_deps = True
            elif not in_deps:
                continue
            elif line[:1].strip() and not line.startswith("#"):
                break                    # next top-level key ends the block
            else:
                # "  - netcdf4>=1.6" -> "netcdf4". Comment lines do not match.
                match = re.match(r"\s*-\s*([A-Za-z0-9_.\-]+)", line)
                if match:
                    names.append(match.group(1).lower())
    return names


def check_environment(verbose=True):
    """Import everything environment.yml asks for; return what is not usable.

    A real import, not importlib.util.find_spec: a package can be installed and
    still fail to load, which is what a mixed conda/pip environment looks like
    from the inside. That case reports BROKEN rather than passing silently.
    """
    problems = []
    for conda_name in required_packages(env_file):
        if conda_name in NOT_IMPORTED:
            continue
        mod_name = IMPORT_NAME.get(conda_name, conda_name)
        try:
            mod = importlib.import_module(mod_name)
        except ImportError:
            problems.append(conda_name)
            if verbose:
                print(f"MISSING  {mod_name}")
        except Exception as exc:          # installed, but will not load
            problems.append(conda_name)
            if verbose:
                print(f"BROKEN   {mod_name:<12} {type(exc).__name__}: {exc}")
        else:
            if verbose:
                print(f"OK       {mod_name:<12} "
                      f"{getattr(mod, '__version__', '?')}")
    return problems


env_file = next((p for p in ENV_CANDIDATES if os.path.exists(p)), None)
if env_file is None:
    raise FileNotFoundError(
        "environment.yml not found. Open this notebook from its own folder, "
        "0.1_Setup_and_Environment, so the working directory is right.")

print(f"Python            : {platform.python_version()}")
print(f"Working directory : {os.getcwd()}")
print(f"Checked against   : {env_file}")
print()

problems = check_environment()

print()
if problems:
    print("Needs attention:", ", ".join(problems))
    print("Run the repair cell below.")
else:
    print("Environment ready - everything environment.yml asks for is here.")

> **On `Working directory`.** Every tutorial notebook writes its data to `os.getcwd()/data/`, which is that notebook's own module folder. There is no path configuration to get wrong — but if the working directory above is not the folder this notebook lives in, your files will end up somewhere surprising.

### 1.1 Repair — only if something above is not `OK`

**Skip this cell if every line said `OK`.** Otherwise run it. It:

1. Finds the conda (or mamba) that owns the environment this notebook is running in.
2. Runs `conda env update` against `environment.yml`, into that same environment.
3. Re-checks, and tells you whether you still need to restart the kernel.

It prints as it goes. Expect a few minutes — most of that is conda solving.

This works the same way in a codespace and on your own machine: it installs into whatever environment the kernel is using, so there is no container rebuild and no environment name to get right.

**Why it uses conda rather than `pip install`.** Cartopy, rasterio and netCDF4 wrap compiled GEOS, PROJ and GDAL libraries. A pip wheel installed on top of a conda environment brings a second copy of those libraries, and the crash that follows is much harder to diagnose than the missing package you started with. The cell falls back to pip only when there is no conda at all — in a plain venv there is no conda copy to clash with.

In [ ]:
# --- Repair: run only if the cell above reported MISSING or BROKEN. --------
# Updates this kernel's own environment from environment.yml, in place. Works
# in a codespace and on a local machine, because it targets sys.prefix rather
# than an environment name.
import shutil
import subprocess
import sys
from pathlib import Path


def find_conda():
    """The conda or mamba executable that owns this kernel's environment.

    sys.prefix is the environment the notebook is running in. A named env lives
    at <base>/envs/<name>, so the base installation is two levels up; in the
    codespace the kernel is in base itself, and the first root is the one that
    matches. Prefixes are checked before PATH: a conda earlier on PATH may well
    belong to a different installation than the one running this kernel.
    """
    roots = [Path(sys.prefix), Path(sys.prefix).parent.parent]
    for exe in ("mamba", "conda"):        # mamba first, it solves far faster
        for root in roots:
            for bindir in ("bin", "Scripts"):        # Scripts on Windows
                for name in (exe, f"{exe}.exe"):
                    candidate = root / bindir / name
                    if candidate.exists():
                        return candidate
        on_path = shutil.which(exe)
        if on_path:
            return Path(on_path)
    conda_exe = os.environ.get("CONDA_EXE", "")      # set by conda shell init
    return Path(conda_exe) if conda_exe and Path(conda_exe).exists() else None


def run_streaming(cmd):
    """Run a command, printing its output into the notebook as it arrives.

    A subprocess that inherits the kernel's stdout writes to the kernel log,
    not to the cell, so a long conda solve would look like a hung notebook.
    """
    print("$", " ".join(str(part) for part in cmd), "\n", flush=True)
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="", flush=True)
    return proc.wait()


problems = check_environment(verbose=False)

if not problems:
    print("Nothing to repair - the environment already matches environment.yml.")
else:
    print("Repairing:", ", ".join(problems), "\n")
    conda = find_conda()

    if conda is not None:
        # --prefix targets this kernel's environment and overrides the name: in
        # environment.yml, so the same command is right in the codespace (base)
        # and locally (eyes-on-earth). No --prune: it would remove anything not
        # in the file, which is not what a student asked for.
        status = run_streaming([str(conda), "env", "update",
                                "--prefix", sys.prefix, "--file", env_file])
    else:
        # No conda anywhere: this is a plain venv, so pip is the right tool and
        # there is no conda copy of the compiled libraries to clash with.
        # matplotlib-base is a conda-only split package; the wheel is matplotlib.
        PIP_NAME = {"matplotlib-base": "matplotlib"}
        print("No conda found - installing with pip instead.\n")
        status = run_streaming([sys.executable, "-m", "pip", "install",
                                *(PIP_NAME.get(p, p) for p in problems)])

    print()
    if status != 0:
        print(f"The install failed (exit code {status}). Read the output above.")
        print("If it mentions permissions or a solve conflict, send it to your")
        print("instructor rather than retrying - see S01 Troubleshooting.")
    else:
        importlib.invalidate_caches()     # let Python see the new files
        remaining = check_environment()
        print()
        if remaining:
            print("Still not importable. Restart the kernel (Kernel > Restart")
            print("Kernel) and run the check cell again: a freshly installed")
            print("compiled package sometimes needs a new interpreter process.")
        else:
            print("Repaired. Carry on - and if a later cell still fails on one")
            print("of these packages, restart the kernel and re-run.")

## 2. Make a test map

This draws coastlines with Cartopy. If you see Southeast Asia below, your environment works.

In a Codespace the coastline data is baked into the image, so this is instant. Elsewhere Cartopy downloads it the first time, which needs a network connection and takes a moment.

In [ ]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 5), subplot_kw={"projection": ccrs.PlateCarree()})
ax.set_extent([90, 140, -15, 25], crs=ccrs.PlateCarree())
ax.add_feature(cfeature.LAND, facecolor="0.9")
ax.add_feature(cfeature.OCEAN, facecolor="#dbeafe")
ax.coastlines(resolution="50m", linewidth=0.6)
ax.gridlines(draw_labels=True, linewidth=0.3, color="0.6")
ax.set_title("Setup check - if you can see this map, you are good to go")
plt.show()

## 3. Check you can read a NetCDF and write a GeoTIFF

Every tutorial does some version of this, so it is worth proving the chain works before you depend on it. No download needed — we make a small array, write it, and read it back.

In [ ]:
import numpy as np
import rioxarray  # registers the .rio accessor
import xarray as xr

da = xr.DataArray(
    np.random.default_rng(0).random((50, 60)).astype("float32"),
    coords={"lat": np.linspace(25, -15, 50), "lon": np.linspace(90, 140, 60)},
    dims=("lat", "lon"),
    name="test",
)

da = da.rio.set_spatial_dims(x_dim="lon", y_dim="lat").rio.write_crs("EPSG:4326")
da.rio.to_raster("setup_check.tif")

check = xr.open_dataarray("setup_check.tif", engine="rasterio")
print("CRS    :", check.rio.crs)
print("Bounds :", check.rio.bounds())
print("Shape  :", check.shape)

os.remove("setup_check.tif")
print("\nGeoTIFF read/write works.")

## Accounts you need

Two tutorials need data-provider logins, and one of them is **not** instant. Set them up before the sessions — work through [0.2.1 Account Check](../0.2_Data_Access_Accounts/0.2.1_Account_Check.ipynb).

| Provider | Needed for | Approval |
|---|---|---|
| NASA Earthdata | PACE (2.1), SWOT (2.3), OSCAR (2.4) | Immediate |
| JAXA P-Tree | Himawari SST (2.2) | **Up to several working days** |

## Next

Go to [Tutorial 2](../2.0_Tutorial_2_Overview_and_Assignment/README.md).